# The Window Dilemma: Why Concept Drift Detecton is Ill-Posed

This notebook contains all of the code used to run the experiments presented in the paper "The Window Dilemma: Why Concept Drift Detection is Ill-Posed" submitted to IDA2026

**Datasets:**
All Datasets are sourced from USP Data Stream Repository: https://sites.google.com/view/uspdsrepository

You should not need to rename the datasets, just make sure they are in the correct directory.

## Code Starts Here...

You may need to install river first. If you do not want this to be installed into your global Python environment, please use a virtual environment.

In [ ]:
!pip install river

In [ ]:
!pip install scikit-learn==1.7.2

In [ ]:
import numpy as np
import pandas as pd
import math

In [ ]:
# Datasets
def read_csv_as_dataset(filename, target, has_header: bool = True):
  if has_header:
    df = pd.read_csv(filename)
  else:
    df = pd.read_csv(filename, header = None)
  labels = df.pop(target)

  return df, labels

def normalize_columns(df, columns):
  # Normalize Data
  for col in columns:
    lo, hi = df[col].min(), df[col].max()
    df[col] = (df[col] - lo) / (hi - lo)

  return df

def standardize_columns(df, columns):
  # Standardize Data
  for col in columns:
    df[col] = (df[col] - df[col].mean()) / df[col].std()

  return df

In [ ]:
# Binary Datasets

def read_Elec2():
  df, labels = read_csv_as_dataset('Electricity.csv', 8, False)
  df.pop(0) # Removes the date from the stream
  #df.pop(1) # Removes the day from the stream
  df[1] /= 7
  return df, labels

def read_NOAA():
  df, labels = read_csv_as_dataset('NOAA.csv', 8,  False)
  labels = labels - 1

  df = standardize_columns(df, df.columns)

  return df, labels

# Read data for Luxembourg
# Data Should Already be normalized
def read_Luxembourg():
  df, labels = read_csv_as_dataset('Luxembourg.csv', 'label')
  # If you want to remove the overly informative at3 feature
  #df.pop('at3')
  df.pop('at31')
  return df, labels

def read_Ozone():
  df, labels = read_csv_as_dataset('Ozone.csv', 72, False)
  labels = labels - 1

  df = standardize_columns(df, df.columns)

  return df, labels

def read_Yoga():
  df, labels = read_csv_as_dataset('Yoga.csv', 426, False)
  labels = labels - 1

  df = standardize_columns(df, df.columns)

  return df, labels

def read_MIRS():
  df, labels = read_csv_as_dataset('MIRS.csv', 3600, False)
  labels = labels - 1

  df = standardize_columns(df, df.columns)

  return df, labels

In [ ]:
def read_CovType():
  df, labels = read_csv_as_dataset('ForestCoverType.csv', 54, False)
  labels = labels - 1

  return df, labels

def read_Chess():
  df, labels = read_csv_as_dataset('Chess.csv', 'label')

  df = standardize_columns(df, df.columns)

  return df, labels

def read_KeyStroke():
  df, labels = read_csv_as_dataset('Keystroke.csv', 10, False)
  labels = labels - 1

  df = standardize_columns(df, df.columns)

  return df, labels

def read_Rialto():
  df, labels = read_csv_as_dataset('Rialto.csv', 27, False)
  df = df.drop(columns=[6,7,11,20]) # These columns are always 0
  df = standardize_columns(df, df.columns)

  return df, labels

def read_InsectsIncremental():
  df, labels = read_csv_as_dataset('INSECTS incremental_balanced.csv', 33, False)
  labels = labels.replace([2,3,4,5,11,12], [0, 1, 2, 3, 4, 5])

  df = standardize_columns(df, df.columns)

  return df, labels

def read_InsectsAbrupt():
  df, labels = read_csv_as_dataset('INSECTS abrupt_balanced.csv', 33, False)
  labels = labels.replace([2,3,4,5,11,12], [0, 1, 2, 3, 4, 5])

  df = standardize_columns(df, df.columns)

  return df, labels


#### Quick Plot Utitlties

In [ ]:
import matplotlib.pyplot as plt

def accuracy_plot(data, labels = None, datasetname = None):
  fig, ax = plt.subplots()
  for i, result in enumerate(data):
    label = labels[i] if labels is not None else None
    label = f'{label} ({round(result[-1], 2)})'
    ax.plot(np.arange(len(result)), result, label=label)

  if labels is not None:
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
  if datasetname is not None:
    plt.title(datasetname)
  plt.show()

def mean_plot(data, labels=None, datasetname=None):
  fig, ax = plt.subplots()
  for i, result in enumerate(data):
    label = labels[i] if labels is not None else None
    label = f'{label} ({round(np.mean(result), 2)})'
    ax.plot(np.arange(len(result)), result, label=label)

  if labels is not None:
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
  if datasetname is not None:
    plt.title(datasetname)
  plt.show()

#### Models

In [ ]:
import river.base as rbase
import river.dummy as rdummy
import river.forest as rforest
import river.naive_bayes as rnb
import river.tree as rtree

import sklearn.ensemble as ensemble
import sklearn.neural_network as neural_network

def make_VFDT(**kwargs):
  return rtree.HoeffdingTreeClassifier(**kwargs)

def make_AdaptiveVFDT(**kwargs):
  return rtree.HoeffdingAdaptiveTreeClassifier(**kwargs)

def make_AMForest(**kwargs):
  return rforest.AMFClassifier(**kwargs)

def make_ARForest(**kwargs):
  seed = 42 if 'seed' not in kwargs else kwargs['seed']
  return rforest.ARFClassifier(seed=seed, **kwargs)

def make_NB():
  return rnb.GaussianNB()

def make_LastClass():
  return rdummy.NoChangeClassifier()

def make_MajorityClass():
  return rdummy.PriorClassifier()

# Non River Models
def make_RandomForest(seed: int = 42, **kwargs):
  return ensemble.RandomForestClassifier(random_state=seed, **kwargs)

**D3 Detector:**

Original Implementation taken from:
https://github.com/ogozuacik/river/blob/master/river/drift/d3.py

This version has been slightly edited to account for newer versions of River.

In [ ]:
from copy import deepcopy

# Note: Other Imports should come from previous code block
import river.metrics as rmetrics


class D3(rbase.DriftDetector):
    r"""Drift Detection Method

    D3 (Discriminative Drift Detector) is an unsupervised drift detection
    method which uses a discriminative classifier that can be used with any
    online algorithm without a built-in drift detector. It holds a fixed
    size sliding window of the latest data having two sets: the old and the
    new. A simple classifier is trained to distinguish these sets. It
    detects a drift with respect to classifier performance (AUC).

    Parameters
    ----------
    window_size
        The size of the data window.
    auc_threshold
        Required AUC score to signal a drift.
        From 0.5 to 1.0
    discriminative_classifier
        Classifier to be used to distinguish old data from the new data.
        Any classifier in the river can be used.

    Notes
    -----
    * This method looks for drifts in multi-dimensional data.
    * It is advised to use a simple model as a discriminative classifier
      since the goal of it is to determine if the old data and the new data
      are seperable, not to classify them.
    * This implementation differs from the original one in the paper,
      making the size of the old and new data windows equal, and allowing
      to use any classifier for discriminating old and new data.


    Examples
    --------
    >>> from river import synth
    >>> from river.drift import D3

    >>> d3 = D3()

    >>> # Simulate a data stream
    >>> data_stream = synth.Hyperplane(
    ...    seed=42, n_features=5, n_drift_features=3, mag_change=0.5)

    >>> # Update drift detector and verify if change is detected
    >>> i = 1
    >>> for x, y in data_stream.take(500):
    ...     in_drift, in_warning = d3.update(x)
    ...     if in_drift:
    ...         print(f"Change detected at index {i}")
    ...     i += 1
    Change detected at index 300

    References
    ----------
    [^1]: Ömer Gözüaçık, Alican Büyükçakır, Hamed Bonab, Fazli Can: Unsupervised concept drift detection with a discriminative classifier. CIKM 2019: 2365-2368

    """

    _AUC_NUM_THRESHOLDS = 20
    _LABEL_FOR_NEW_DATA = True
    _LABEL_FOR_OLD_DATA = False

    def __init__(
        self,
        window_size=200,
        auc_threshold=0.7,
        discriminative_classifier=rtree.HoeffdingTreeClassifier(
            grace_period=40, max_depth=3
        ),
    ):
        super().__init__()
        self.auc_threshold = auc_threshold
        self.sub_window_size = int(window_size / 2)
        self.discriminative_classifier = discriminative_classifier
        self.old_data_window = [None] * self.sub_window_size
        self.new_data_window = [None] * self.sub_window_size
        self.data_labels = None
        self.store_labels = False
        self.old_data_window_index = 0
        self.new_data_window_index = 0
        self.auc = rmetrics.ROCAUC(n_thresholds=D3._AUC_NUM_THRESHOLDS)

    def current_data_and_labels(self):
        """Returns the data and labels for the current data window.

        They can be used when for retraining the stream classifier after drift detection.
        """
        return self.old_data_window, self.data_labels

    def update_labels_if_storing_enabled(self, index, label):
        """Update the labels array if storing is enabled"""
        if not self.store_labels:
            return
        self.data_labels[index] = label

    def reset(self):
        """Reset the change detector."""
        self.old_data_window = [None] * self.sub_window_size
        self.new_data_window = [None] * self.sub_window_size
        self.old_data_window_index = 0
        self.new_data_window_index = 0
        self.data_labels = None
        self.store_labels = False
        self.auc = rmetrics.ROCAUC(n_thresholds=D3._AUC_NUM_THRESHOLDS)
        self.discriminative_classifier = self.discriminative_classifier.clone()

    def update(self, sample, label=None):
        """Update the change detector with a single sample.

        Parameters
        ----------
        sample
            An instance from the data stream (dict) having N items (number of features).
        label
            Class label for sample.

        Notes
        -----
        * The label is is not used in the detection process. It can be set to None.
        * If the label is not set to None, D3 will store the last window_size/2
          class labels of the samples. It is useful for retraining the stream classifier
          when a drift is detected.
        """
        if self.drift_detected:
            self._drift_detected = False

        # Start storing labels if not None
        if (label is not None) and (self.store_labels is False):
            self.data_labels = [None] * self.sub_window_size
            self.store_labels = True

        if self.old_data_window_index < self.sub_window_size:
            self.old_data_window[self.old_data_window_index] = sample
            self.old_data_window_index += 1
            return self.drift_detected

        self.new_data_window[self.new_data_window_index] = sample
        self.update_labels_if_storing_enabled(self.new_data_window_index, label)

        # Updating discriminative classifier with a sample from the old and new data
        old_data_sample = self.old_data_window[self.new_data_window_index]
        self.discriminative_classifier.learn_one(sample, D3._LABEL_FOR_NEW_DATA)
        self.discriminative_classifier.learn_one(
            old_data_sample, D3._LABEL_FOR_OLD_DATA
        )

        # Update AUC
        prob_new = self.discriminative_classifier.predict_proba_one(sample)[1]
        prob_old = self.discriminative_classifier.predict_proba_one(old_data_sample)[1]
        if prob_new is not None:
          self.auc.update(D3._LABEL_FOR_NEW_DATA, prob_new)

        if prob_old is not None:
          self.auc.update(D3._LABEL_FOR_OLD_DATA, prob_old)

        self.new_data_window_index += 1

        if self.new_data_window_index == self.sub_window_size:
            auc_score = self.auc.get()
            if (auc_score > self.auc_threshold) or (
                auc_score < self.auc_threshold - 0.5
            ):
                self._drift_detected = True
            self.old_data_window = deepcopy(self.new_data_window)
            self.new_data_window_index = 0
            self.auc = rmetrics.ROCAUC(n_thresholds=D3._AUC_NUM_THRESHOLDS)
            self.discriminative_classifier = self.discriminative_classifier.clone()

        return self.drift_detected

**Image-based Drift Detector (IBDD) and Nearest Neighbour-based Density Variation Identification (NNDVI):**

Orignal code taken from: https://github.com/DFKI-NI/unsupervised-concept-drift-detection/tree/main

It has been modified to support our river-style setup.

In [ ]:
import time
from abc import ABC, abstractmethod
from typing import Optional

class UnsupervisedDriftDetector(ABC):
    """
    This abstract base class provides a consistent interface for all unsupervised concept drift detectors.
    """

    def __init__(self, seed: Optional[int] = None):
        if seed is None:
            seed = int(time.time())
        self.seed = seed

    @abstractmethod
    def update(
        self,
        features: dict,
    ) -> bool:
        raise NotImplementedError("This abstract base class does not implement update.")

In [ ]:
import itertools
from collections import deque

class ImageBasedDriftDetector(UnsupervisedDriftDetector):
    """
    Image-Based Drift Detector (IBBD) detects concept drifts by calculating the mean squared deviation of the reference
    data window and the recent data window. If the deviation exceeds thresholds, a drift is signalled. The thresholds
    are determined from the recent deviations and are updated regularly and everytime a drift is detected. Since the
    both deviations and the thresholds are calculated from the initial reference data, the reference data is never
    deleted.

    Source: Souza, V. M. A.; Parmezan, A. R. S.; Chowdhury, F. A.; Mueen, A. (2021). Efficient unsupervised drift
        detector for fast and high-dimensional data streams. Knowledge and Information Systems. Springer Link.
    """

    def __init__(
        self,
        n_samples: int = 300,
        n_consecutive_deviations: int = 1,
        n_permutations: int = 20,
        update_interval: int = 50,
        seed: Optional[int] = None,
    ):
        """
        Init a new IBDD instance.

        :param n_samples: the number of samples stored by both the reference data window and the recent data window
        :param n_consecutive_deviations: the number of consecutive values exceeding thresholds that must be detected
            before a concept drift is signalled
        :param n_permutations: the number of times the reference data is permuted to determine initial thresholds
        :param update_interval: the number of time steps between each update of the thresholds
        """
        super().__init__(seed)
        self.n_samples = n_samples
        self.reference_data = []
        self.recent_data = deque(maxlen=n_samples)
        self.recent_deviations = deque(maxlen=update_interval)
        self.n_consecutive_deviations = n_consecutive_deviations
        self.upper_threshold = None
        self.lower_threshold = None
        self.threshold_diffs = []
        self.update_interval = update_interval
        self.n_permutations = n_permutations
        self.time_step = 0
        self.last_threshold_update = 0
        self.rng = np.random.default_rng(self.seed)

        # Added
        self.drift_detected = False

    def update(self, features: dict) -> bool:
        """
        Update the detector with the given features.

        :param features: the features
        :return: True if a drift occurred, else False
        """
        features = np.fromiter(features.values(), dtype=float)

        self.drift_detected = False

        if self.upper_threshold is None and self.lower_threshold is None:
            self.reference_data.append(features)
            if len(self.reference_data) == self.n_samples:
                self._calculate_initial_thresholds()
        self.recent_data.append(features)
        if (
            len(self.reference_data) == self.n_samples
            and len(self.recent_data) == self.n_samples
        ):
            deviation = self._calculate_mean_squared_deviation(
                np.array(self.recent_data)
            )
            self.recent_deviations.append(deviation)
            if self.time_step - self.last_threshold_update > self.update_interval:
                self._update_thresholds()

            self.drift_detected = self._detect_drift(deviation)
        self.time_step += 1
        return self.drift_detected

    def _detect_drift(self, deviation: float):
        """
        Detect if a concept drift occurred and update the upper and lower thresholds accordingly.

        :param deviation: the most recent mean squared deviation
        :return: True if a drift occurred, else False
        """
        evaluation_values = np.fromiter(
            itertools.islice(
                self.recent_deviations,
                len(self.recent_deviations) - (self.n_consecutive_deviations + 1),
                len(self.recent_deviations),
            ),
            dtype=float,
        )
        if np.all(evaluation_values >= self.upper_threshold):
            self.upper_threshold = deviation + np.std(self.recent_deviations)
            self.lower_threshold = deviation - np.mean(self.threshold_diffs)
            self.threshold_diffs.append(self.upper_threshold - self.lower_threshold)
            self.last_threshold_update = self.time_step
            return True
        elif np.all(evaluation_values <= self.lower_threshold):
            self.lower_threshold = deviation - np.std(self.recent_deviations)
            self.upper_threshold = deviation + np.mean(self.threshold_diffs)
            self.threshold_diffs.append(self.upper_threshold - self.lower_threshold)
            self.last_threshold_update = self.time_step
            return True
        return False

    def _calculate_mean_squared_deviation(self, other_data: np.array) -> float:
        """
        Calculate the mean squared deviation of the data stored in the reference window and the given other data.

        :param other_data: the data compared to the reference data
        :return: the mean squared deviation
        """
        reference_data = np.array(self.reference_data)
        if reference_data.shape != other_data.shape:
            raise ValueError(
                f"Shapes of compared data windows do not match: {reference_data.shape} != {other_data.shape}"
            )

        summands = (reference_data - other_data) ** 2
        return float(np.mean(summands))

    def _update_thresholds(self):
        """
        Update the upper and lower thresholds signalling the presence of a concept drift.
        """
        self.lower_threshold = np.mean(self.recent_deviations) - 2 * np.std(
            self.recent_deviations
        )
        self.upper_threshold = np.mean(self.recent_deviations) + 2 * np.std(
            self.recent_deviations
        )
        self.threshold_diffs.append(self.upper_threshold - self.lower_threshold)
        self.last_threshold_update = self.time_step

    def _calculate_initial_thresholds(self):
        """
        Calculate the initial upper and lower concept drift thresholds from expected deviations. The expected deviations
        are determined by evaluating permutations of the reference data.

        :raise: ValueError if either threshold is not None
        """
        if self.upper_threshold is not None or self.lower_threshold is not None:
            raise ValueError(
                "This method is intended for the calculation of initial thresholds only"
            )

        indices = np.arange(self.n_samples)
        for _ in range(self.n_permutations):
            self.rng.shuffle(indices)
            msd = self._calculate_mean_squared_deviation(
                np.array(self.reference_data)[indices]
            )
            self.recent_deviations.append(msd)
        deviations = np.fromiter(self.recent_deviations, dtype=float)
        self.lower_threshold = np.mean(deviations) - 2 * np.std(deviations)
        self.upper_threshold = np.mean(deviations) + 2 * np.std(deviations)
        self.threshold_diffs.append(self.upper_threshold - self.lower_threshold)

**Wrappers:**

In [ ]:
import river.drift as rdrift

class RiverClassifierMimic: # Mimics River's predict one, learn one
  def predict_one(self, x):
    raise NotImplementedError()

  def learn_one(self, x, y):
    raise NotImplementedError()

class RiverDetectorMimic: # Mimics River's update
  def update(self, x):
    raise NotImplementedError()

class DriftResetClassifier(RiverClassifierMimic): # Base Class for Reset Strategy
  def __init__(self, model_class, detector_class, model_kwargs, detector_kwargs,
               reset_detector: bool = False):
    self.model_class = model_class
    self.detector_class = detector_class
    self.model_kwargs = model_kwargs
    self.detector_kwargs = detector_kwargs
    self._reset_detector = reset_detector

    self._reset(True) # Initialize

  def _reset(self, force: bool = False):
    self.model = self.model_class(**self.model_kwargs)

    if force or self._reset_detector:
      self.detector = self.detector_class(**self.detector_kwargs)

    self.drift_detected = self.detector.drift_detected

  def predict_one(self, x):
     return self.model.predict_one(x)

class PerformanceDriftResetClassifier(DriftResetClassifier):
  def learn_one(self, x, y):

    if self.drift_detected:  # Reset if Drift was detected on previous instance
      self._reset()

    signal = 1 if y != self.model.predict_one(x) else 0 # 1 for error 0 for correct

    self.model.learn_one(x, y)
    self.detector.update(signal)
    self.drift_detected = self.detector.drift_detected

    return self

class UnsupervisedDriftResetClassifier(DriftResetClassifier):
  def learn_one(self, x, y):

    if self.drift_detected:  # Reset if Drift was detected on previous instance
      self._reset()

    self.model.learn_one(x, y)
    self.detector.update(x)  # We store entire instance
    self.drift_detected = self.detector.drift_detected

    return self

class ResetEveryNClassifier(RiverClassifierMimic):
  def __init__(self, model_class, model_kwargs, N: int):
     self.model_class = model_class
     self.model_kwargs = model_kwargs
     self.N = N

     self._reset()  # Initialize

  def _reset(self):
    self.model = self.model_class(**self.model_kwargs)
    self.counter = 0

  def predict_one(self, x):
     return self.model.predict_one(x)

  def learn_one(self, x, y):

    if self.counter == self.N:
      self._reset()

    self.model.learn_one(x, y)
    self.counter += 1

    return self

class BufferEveryNClassifier(RiverClassifierMimic):
  def __init__(self, model, N : int) -> None:
    self.model = model
    self.counter = 0
    self.N = N

    self.mc = rdummy.PriorClassifier()

    self.is_fit = False

    self.instance_buffer = []
    self.label_buffer = []

  def predict_one(self, x):
    if self.is_fit:
      return self.model.predict_one(x)
    else:
      return self.mc.predict_one(x)

  def learn_one(self, x, y):

    self.counter += 1
    self.instance_buffer.append(x)
    self.label_buffer.append(y)

    if not self.is_fit:
      self.mc.learn_one(x, y)

    if self.counter == self.N:
      for i in range(self.N):
        self.model.learn_one(self.instance_buffer[i], self.label_buffer[i])

      self.instance_buffer.clear()
      self.label_buffer.clear()
      self.counter = 0

      self.is_fit = True

    return self


In [ ]:
def make_PerformanceDriftResetClassifier(model_class, detector_class,
                                         model_kwargs, detector_kwargs,
                                         reset_detector = False):
  return PerformanceDriftResetClassifier(
        model_class,
        detector_class,
        model_kwargs,
        detector_kwargs,
        reset_detector
      )

def make_UnsupervisedDriftResetClassifier(model_class, detector_class,
                                         model_kwargs, detector_kwargs,
                                         reset_detector = False):
  return UnsupervisedDriftResetClassifier(
        model_class,
        detector_class,
        model_kwargs,
        detector_kwargs,
        reset_detector
      )

def make_ResetEveryNClassifier(model_class, model_kwargs, N):
  return ResetEveryNClassifier(model_class, model_kwargs, N)

def make_BufferEveryNClassifier(model, N):
  return BufferEveryNClassifier(model, N)

**Batch Learners:**

In [ ]:
from collections import deque
import river.dummy as rdummy

class RetrainEveryNClassifier(RiverClassifierMimic):
  def __init__(self, base_model, N: int = 100, max_samples: int = None,
               should_retrain: bool = True):
    self.base_model = base_model

    self.N = N
    self.counter = 0

    # While building training sample, we use MajorityClass
    self.is_fit = False
    self.should_retrain = should_retrain
    self.mc = rdummy.PriorClassifier()

    self.samples = deque([], maxlen=max_samples) if max_samples is not None else []
    self.labels = deque([], maxlen=max_samples) if max_samples is not None else []

  def _dict_to_numpy(self, x: dict):
    return np.fromiter(x.values(), dtype=float).reshape(1, -1)

  def _refit(self):

    self.counter = 0

    X = np.vstack(self.samples)
    Y = np.hstack(self.labels)

    if len(np.unique(Y)) == 1:
      return

    self.base_model.fit(X, Y)
    self.is_fit = True

  def predict_one(self, x):
    if self.is_fit:
      return self.base_model.predict(self._dict_to_numpy(x))[0]
    else:
      return self.mc.predict_one(x)

  def learn_one(self, x, y):
    self.counter +=1
    self.mc.learn_one(x, y)

    self.samples.append(self._dict_to_numpy(x))
    self.labels.append(y)

    if self.counter == self.N:
      if not self.is_fit or self.should_retrain:
        self._refit()

    return self

In [ ]:
def make_RetrainEveryNClassifier(model, **kwargs):
  return RetrainEveryNClassifier(model, **kwargs)

#### Test-then-train Loop

In [ ]:
import json
import river.metrics as rmetrics

# Assumes model is river compatible
def test_then_train(df, labels, model : rbase.Classifier):

  acc = rmetrics.Accuracy()
  cm = rmetrics.ConfusionMatrix()
  kappa = rmetrics.CohenKappa()

  ACC = []
  KAPPA = []
  FPS = []
  FNS = []

  # Convert df to dict (needed for river)
  X = df.to_dict(orient='records')
  for i, xi in enumerate(X):
    yi = labels[i]

    # Test
    yi_hat = model.predict_one(xi)

    # Train
    model.learn_one(xi, yi)

    # Report Metrics
    if yi_hat is not None:
      acc.update(yi, yi_hat)
      cm.update(yi, yi_hat)
      kappa.update(yi, yi_hat)

    ACC.append(acc.get())
    KAPPA.append(kappa.get())
    FPS.append(cm.total_false_positives)
    FNS.append(cm.total_false_negatives)


  return ACC, KAPPA, FPS, FNS


def save_results(results, filename="divergence_results.json"):
    with open(filename, "w") as f:
        json.dump(results, f, indent=4)

#### Run Experiments (base)

In [ ]:
DATASET_NAME = 'Keystroke_stream' # Choose filename for saved results
df, labels = read_KeyStroke() # Choose dataset you want to load.
models = {
    'Last Class': make_LastClass(),
    'Majority Class': make_MajorityClass(),
    'Naive Bayes': make_NB(),
    '(DDM) Naive Bayes': make_PerformanceDriftResetClassifier(
        make_NB,
        rdrift.binary.DDM,
        {},{}
        ),
    '(ADWIN) Naive Bayes': make_PerformanceDriftResetClassifier(
        make_NB,
        rdrift.ADWIN,
        {},{}
        ),
    'Reset (N=60) Naive Bayes': make_ResetEveryNClassifier(make_NB, {}, 60),
    'Hoeffding Tree': make_VFDT(),
    '(DDM) Hoeffding Tree': make_PerformanceDriftResetClassifier(
        make_VFDT,
        rdrift.binary.DDM,
        {},{}
        ),
    '(ADWIN) Hoeffding Tree': make_PerformanceDriftResetClassifier(
        make_VFDT,
        rdrift.ADWIN,
        {},{}
        ),
    'Reset (N=60) Hoeffding Tree': make_ResetEveryNClassifier(make_VFDT, {}, 60),
    'Hoeffding Adaptive Tree': make_AdaptiveVFDT(),
    'Aggregated Mondrian Forest' : make_AMForest(),
    'Adaptive Random Forest': make_ARForest()
    }

results = {}
data = []

for model_name in models:
  acc, kappa, fps, fns = test_then_train(df, labels, models[model_name])

  results[model_name] = {
      'acc': acc,
      'kappa': kappa,
      'fps': fps,
      'fns': fns
  }

  data.append(acc)

save_results(results, filename=f'{DATASET_NAME}.json')

accuracy_plot(data, [model for model in models])

In [ ]:
accuracy_plot([results[model]['kappa'] for model in results], [model for model in results])

#### Run Experiments (Additional)

Contains additional experiments that can't be conducted with river alone.

In [ ]:
import river.linear_model as rlinear_model
import math

DATASET_NAME = 'MIRS_additional' # Filename for saved results
df, labels = read_MIRS() # Dataset to load.

D3_LR_KWARGS = {
    "discriminative_classifier": rlinear_model.LogisticRegression()
}

models = {
   '(D3-LR) Naive Bayes': make_UnsupervisedDriftResetClassifier(
        make_NB,
        D3,
        {}, D3_LR_KWARGS
        ),
    '(D3-HT) Naive Bayes': make_UnsupervisedDriftResetClassifier(
        make_NB,
        D3,
        {},{}
        ),
    '(D3-LR) Hoeffding Tree': make_UnsupervisedDriftResetClassifier(
        make_VFDT,
        D3,
        {}, D3_LR_KWARGS
        ),
    '(D3-HT) Hoeffding Tree': make_UnsupervisedDriftResetClassifier(
        make_VFDT,
        D3,
        {},{}
        ),
    '(IBDD) Naive Bayes': make_UnsupervisedDriftResetClassifier(
        make_NB,
        ImageBasedDriftDetector,
        {}, {}
        ),
    '(IBDD) Hoeffding Tree': make_UnsupervisedDriftResetClassifier(
        make_VFDT,
        ImageBasedDriftDetector,
        {},{}
        ),
    '(NNDVI) Naive Bayes': make_UnsupervisedDriftResetClassifier(
        make_NB,
        NNDVI,
        {}, {}
        ),
    '(NNDVI) Hoeffding Tree': make_UnsupervisedDriftResetClassifier(
        make_VFDT,
        NNDVI,
        {},{}
        )
   '(N=100, No Retraining) Random Forest': make_RetrainEveryNClassifier(
        make_RandomForest(),
        N=100,
        should_retrain=False
        ),
   '(N=50, Reset Train) Random Forest': make_RetrainEveryNClassifier(
        make_RandomForest(),
        N=50,
        max_samples=50
        ),
   '(N=50, Incremental) Random Forest': make_RetrainEveryNClassifier(
        make_RandomForest(),
        N=50
        )
    }

results = {}
data = []

for model_name in models:
  print(model_name)
  acc, kappa, fps, fns = test_then_train(df, labels, models[model_name])

  results[model_name] = {
      'acc': acc,
      'kappa': kappa,
      'fps': fps,
      'fns': fns
  }

  data.append(acc)

save_results(results, filename=f'{DATASET_NAME}.json')

accuracy_plot(data, [model for model in models])

#### ARF and AMF (Buffer) Additional Experiments

Supplementary experiments for the ARF and AMF models that follow the same update cadence as the batch learners

In [ ]:
DATASET_NAME = 'Rialto_ARF_buffer' # Filename to save results to
df, labels = read_Rialto() # Dataset to load

N = 50 # Update / Learning Frequency

models = {
   f'(Buffer={N}) Adaptive Random Forest': make_BufferEveryNClassifier(
          make_ARForest(),
          N
        ),
   f'(Buffer={N}) Aggregated Mondrian Forest': make_BufferEveryNClassifier(
          make_AMForest(),
          N
        )
  }

results = {}
data = []

for model_name in models:
  print(model_name)
  acc, kappa, fps, fns = test_then_train(df, labels, models[model_name])

  results[model_name] = {
      'acc': acc,
      'kappa': kappa,
      'fps': fps,
      'fns': fns
  }

  data.append(acc)
  print(np.mean(acc))

save_results(results, filename=f'{DATASET_NAME}.json')
mean_plot(data, [model for model in models])

#### Window Dilemma Images

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rc('font', size=18)

# Parameters
k = 75 # Specify where you want the cutoff point to be.
n_samples_per_t = 1000

# Time windows
t1 = np.arange(0, k)
t2 = np.arange(k, 100)

# Generate Samples
samples1 = np.concatenate([
    np.random.normal(loc=t/10, scale=1, size=n_samples_per_t) for t in t1
])
samples2 = np.concatenate([
    np.random.normal(loc=t/10, scale=1, size=n_samples_per_t) for t in t2
])

# KDE implementation (Gaussian kernel)
def kde_gaussian(data, grid, bandwidth):
    return np.mean(
        np.exp(-0.5 * ((grid[:, None] - data[None, :]) / bandwidth) ** 2)
        / (bandwidth * np.sqrt(2 * np.pi)),
        axis=1
    )

# Grid Over Data Range
xmin = min(samples1.min(), samples2.min())
xmax = max(samples1.max(), samples2.max())
grid = np.linspace(xmin, xmax, 400)

# Bandwidth using Silverman's Rule
def silverman_bandwidth(x):
    return 1.06 * np.std(x) * (len(x) ** (-1/5))

bw1 = silverman_bandwidth(samples1)
bw2 = silverman_bandwidth(samples2)

kde1 = kde_gaussian(samples1, grid, bw1)
kde2 = kde_gaussian(samples2, grid, bw2)

# Plot KDE Curves
plt.figure(figsize=(10,4))
plt.plot(grid, kde1, label="$\\mathcal{D}_{W(0,75)}$", lw=2,
         color='#5D3A9B')
plt.fill_between(grid, kde1, color='#5D3A9B', alpha=0.5)
plt.plot(grid, kde2, label="$\\mathcal{D}_{W(75,100)}$", lw=2,
         color='#E66100', linestyle='--')
plt.fill_between(grid, kde2, color='#E66100', alpha=0.5, hatch='/')

ax = plt.gca()
ax.set_ylim([0.0, 0.5])
ax.set_xlim([-5.0, 15.0])

plt.xticks([-5.0, 0.0, 5.0, 10.0, 15.0])
plt.yticks([0.0, 0.25, 0.5])

plt.legend(loc="upper center", ncol=2, fontsize='large')
plt.xlabel("Value", fontsize='large')
plt.ylabel("Density", fontsize='large')
plt.show()